# 📄 LLM File System Assistant — Project Workbook

This notebook demonstrates a complete end-to-end implementation of an **LLM-powered File System Assistant** that uses **function calling / tool use** to interact with resume documents.

---

## 🗂️ Project Overview

| Component | Description |
|---|---|
| **Part A** | Core File System Tools (`read_file`, `list_files`, `write_file`, `search_in_file`) |
| **Part B** | LLM Integration with function calling (OpenAI / OpenRouter API) |

### Learning Objectives
- Understand LLM function calling / tool use patterns
- Implement structured tool interfaces with JSON schemas
- Handle file I/O operations programmatically (PDF, DOCX, TXT)
- Parse and validate documents

### Files in the Project
```
llm-file-system-assistant/
├── fs_tools.py             # Core file system tools (Part A)
├── llm_file_assistant.py   # LLM integration & function calling (Part B)
├── requirements.txt        # Dependencies
├── .env                    # API key configuration
├── README.md               # Project documentation
├── workbook.ipynb          # This notebook
└── resumes/                # Sample PDF resume files
    ├── resume_john_doe.pdf
    ├── resume_bob_williams.pdf
    └── resume_eva_davis.pdf
```

---
## ⚙️ Step 1 — Dependencies Setup

All required Python packages (`openai`, `python-dotenv`, `pypdf`, `python-docx`, `reportlab`) are specified in `requirements.txt`. Ensure they are installed in your environment before running this notebook:

```bash
pip install -r requirements.txt
```

---
## 📦 Step 2 — Import Libraries

Import all third-party and standard-library modules used throughout the project.

In [7]:
import os
import json
import datetime
from pathlib import Path

# PDF parsing
from pypdf import PdfReader

# DOCX parsing
import docx

# PDF generation
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# LLM API
from dotenv import load_dotenv
from openai import OpenAI

print("All libraries imported successfully.")

All libraries imported successfully.


---
## 🔑 Step 3 — Configure API Keys

Load environment variables from the `.env` file. 
Ensure your `.env` contains one of the following:
```ini
OPENROUTER_API_KEY=sk-or-v1-...
# OR
OPENAI_API_KEY=sk-...
```

In [8]:
load_dotenv()

OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")

if OPENROUTER_KEY:
    print("Using: OpenRouter API")
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_KEY
    )
    MODEL = os.environ.get("MODEL_NAME", "openai/gpt-4o-mini")
elif OPENAI_KEY:
    print("Using: OpenAI API")
    client = OpenAI(api_key=OPENAI_KEY)
    MODEL = os.environ.get("MODEL_NAME", "gpt-4o-mini")
else:
    raise EnvironmentError("No API key found. Please set OPENROUTER_API_KEY or OPENAI_API_KEY in your .env file.")

print(f"Model: {MODEL}")

Using: OpenRouter API
Model: openai/gpt-4o-mini


---
## 🛠️ Part A — Core File System Tools

Define the four core tool functions that will be exposed to the LLM. Each returns structured dictionaries for predictable, machine-readable responses.

### Tool 1: `read_file(filepath)` — Read Resume Documents

Reads a file (`.pdf`, `.txt`, `.docx`) and extracts:
- Full text content
- Metadata: name, path, size, type, modification date
- Graceful error handling for missing or unsupported files

In [9]:
def read_file(filepath: str) -> dict:
    """
    Read a resume file (PDF, TXT, DOCX) and extract text content along with metadata.

    Args:
        filepath (str): Path to the file to read.

    Returns:
        dict: {'status', 'content', 'metadata'} on success,
              {'status', 'error'} on failure.
    """
    try:
        path = Path(filepath)
        if not path.exists():
            return {"status": "failed", "error": f"File not found: {filepath}"}
        if not path.is_file():
            return {"status": "failed", "error": f"Path is not a file: {filepath}"}

        extension = path.suffix.lower()
        content = ""

        if extension == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
        elif extension == ".pdf":
            reader = PdfReader(str(path))
            content = "\n".join(
                page.extract_text() for page in reader.pages if page.extract_text()
            )
        elif extension == ".docx":
            doc_obj = docx.Document(str(path))
            content = "\n".join(p.text for p in doc_obj.paragraphs)
        else:
            return {
                "status": "failed",
                "error": f"Unsupported file type: '{extension}'. Supported: .pdf, .txt, .docx"
            }

        stat = path.stat()
        mod_time = datetime.datetime.fromtimestamp(
            stat.st_mtime, tz=datetime.timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC")

        return {
            "status": "success",
            "content": content,
            "metadata": {
                "name": path.name,
                "filepath": str(path),
                "size_bytes": stat.st_size,
                "type": extension,
                "modified_date": mod_time
            }
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Read a PDF resume ---
result = read_file("resumes/resume_john_doe.pdf")
print(f"Status  : {result['status']}")
print(f"File    : {result['metadata']['name']}")
print(f"Size    : {result['metadata']['size_bytes']} bytes")
print(f"Modified: {result['metadata']['modified_date']}")
print(f"\nContent Preview (first 300 chars):\n{result['content'][:300]}")

Status  : success
File    : resume_john_doe.pdf
Size    : 2239 bytes
Modified: 2026-07-23 06:44:40 UTC

Content Preview (first 300 chars):
John Doe - Senior Software Engineer
Professional Summary:
Experienced Software Engineer specializing in backend infrastructure, REST API design, and
distributed systems. Expert in Python, FastAPI, and Docker.
Technical Skills:
Python, FastAPI, Docker, PostgreSQL, AWS, Redis, Git
Work Experience:
 S


### Tool 2: `list_files(directory, extension)` — List Directory Contents

Lists all files in a directory with:
- Name, path, size, modified date, type
- Optional extension filter (e.g. `.pdf`, `.txt`)

In [10]:
def list_files(directory: str, extension: str = None) -> list:
    """
    List all files in a directory, optionally filtered by extension.

    Args:
        directory (str): Directory path to list files from.
        extension (str, optional): Extension filter e.g. '.pdf' or 'pdf'.

    Returns:
        list: List of file metadata dicts, or error dict in a list.
    """
    try:
        path = Path(directory)
        if not path.exists() or not path.is_dir():
            return [{"status": "failed", "error": f"Directory not found: {directory}"}]

        ext = None
        if extension:
            ext = extension.strip().lower()
            if not ext.startswith("."):
                ext = f".{ext}"

        files = []
        for item in sorted(path.iterdir()):
            if item.is_file():
                if ext and item.suffix.lower() != ext:
                    continue
                stat = item.stat()
                mod_time = datetime.datetime.fromtimestamp(
                    stat.st_mtime, tz=datetime.timezone.utc
                ).strftime("%Y-%m-%d %H:%M:%S UTC")
                files.append({
                    "name": item.name,
                    "path": str(item),
                    "size_bytes": stat.st_size,
                    "modified_date": mod_time,
                    "type": item.suffix.lower()
                })
        return files
    except Exception as e:
        return [{"status": "failed", "error": str(e)}]


# --- Demo: List all resumes ---
files = list_files("resumes")
print(f"Found {len(files)} resume files:\n")
for f in files:
    print(f"  {f['name']:35s}  {f['size_bytes']:>6} bytes   {f['modified_date']}")

Found 11 resume files:

  resume_alice_harper.pdf                2144 bytes   2026-07-23 06:27:42 UTC
  resume_bob_williams.pdf                2122 bytes   2026-07-23 06:44:40 UTC
  resume_clara_bennett.pdf               2126 bytes   2026-07-23 06:27:42 UTC
  resume_david_miller.pdf                2086 bytes   2026-07-23 06:44:40 UTC
  resume_elena_rostova.pdf               2113 bytes   2026-07-23 06:27:42 UTC
  resume_eva_davis.pdf                   2146 bytes   2026-07-23 06:44:40 UTC
  resume_jane_smith.pdf                  2208 bytes   2026-07-23 06:44:40 UTC
  resume_john_doe.pdf                    2239 bytes   2026-07-23 06:44:40 UTC
  resume_marcus_vane.pdf                 2129 bytes   2026-07-23 06:27:42 UTC
  resume_samuel_brooks.pdf               2116 bytes   2026-07-23 06:27:42 UTC
  summary_john_doe.txt                    814 bytes   2026-07-24 16:49:55 UTC


### Tool 3: `write_file(filepath, content)` — Write Files to Disk

Writes text content to any file path:
- Creates parent directories automatically
- Returns success status, path, and file size

In [11]:
def write_file(filepath: str, content: str) -> dict:
    """
    Write text content to a file, creating parent directories if necessary.

    Args:
        filepath (str): Destination file path.
        content (str): Text content to write.

    Returns:
        dict: {'status', 'message', 'filepath', 'size_bytes'} on success.
    """
    try:
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        stat = path.stat()
        return {
            "status": "success",
            "message": f"Successfully written to {filepath}",
            "filepath": str(path),
            "size_bytes": stat.st_size
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Write a summary file ---
sample_content = """RESUME SUMMARY — John Doe
Role   : Senior Software Engineer
Skills : Python, FastAPI, Docker, AWS, PostgreSQL
Status : Strong Python background — recommend for interview.
"""

result = write_file("summaries/john_doe_summary.txt", sample_content)
print(f"Status  : {result['status']}")
print(f"Written : {result['filepath']}")
print(f"Size    : {result['size_bytes']} bytes")

Status  : success
Written : summaries\john_doe_summary.txt
Size    : 179 bytes


### Tool 4: `search_in_file(filepath, keyword)` — Keyword Search with Context

Searches a document for a keyword (case-insensitive) and returns:
- Line number of each match
- The matched line
- Surrounding context (1 line before and after)

In [12]:
def search_in_file(filepath: str, keyword: str) -> dict:
    """
    Search for a keyword in a file (case-insensitive) with surrounding context.

    Args:
        filepath (str): File path to search in.
        keyword (str): Keyword or phrase to search for.

    Returns:
        dict: {'status', 'keyword', 'matches_found', 'matches'} on success.
    """
    try:
        read_result = read_file(filepath)
        if read_result.get("status") == "failed":
            return read_result

        content = read_result.get("content", "")
        lines = content.splitlines()
        keyword_lower = keyword.lower()

        matches = []
        for i, line in enumerate(lines):
            if keyword_lower in line.lower():
                start = max(0, i - 1)
                end = min(len(lines), i + 2)
                matches.append({
                    "line_number": i + 1,
                    "match": line.strip(),
                    "context": "\n".join(lines[start:end])
                })

        return {
            "status": "success",
            "filepath": str(filepath),
            "keyword": keyword,
            "matches_found": len(matches),
            "matches": matches
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Search for 'Python' in John Doe's resume ---
result = search_in_file("resumes/resume_john_doe.pdf", "Python")
print(f"Keyword      : '{result['keyword']}'")
print(f"File         : {result['filepath']}")
print(f"Matches Found: {result['matches_found']}\n")

for m in result["matches"]:
    print(f"  Line {m['line_number']}: {m['match']}")

Keyword      : 'Python'
File         : resumes/resume_john_doe.pdf
Matches Found: 5

  Line 4: distributed systems. Expert in Python, FastAPI, and Docker.
  Line 6: Python, FastAPI, Docker, PostgreSQL, AWS, Redis, Git
  Line 8:  Senior Python Developer - TechCorp Innovations (2021 - Present)
  Line 9: Architected high-throughput microservices using Python and FastAPI. Reduced API latency by 40%
  Line 12: Built containerized data pipelines with Python and Docker on AWS ECS.


---
## 🤖 Part B — LLM Function Calling Integration

Define the OpenAI-compatible JSON schemas for each tool and wire them into an LLM-driven agentic loop.
The LLM autonomously decides **which tools to call, in what order, and with what arguments** based on a user query.

### Step 5 — Define Tool JSON Schemas

Each tool is described in an OpenAI function calling schema — this tells the LLM exactly what parameters each tool accepts, so it can invoke them correctly.

In [13]:
# OpenAI-compatible tool schemas for all 4 file system functions

tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a resume file (.pdf, .txt, .docx) and extract text content along with metadata.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The absolute or relative file path to read."
                    }
                },
                "required": ["filepath"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all files in a directory, optionally filtering by extension (e.g. '.pdf', '.txt', '.docx').",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {
                        "type": "string",
                        "description": "The directory path to scan (e.g. 'resumes')."
                    },
                    "extension": {
                        "type": "string",
                        "description": "Optional file extension filter (e.g., '.pdf', '.txt', '.docx')."
                    }
                },
                "required": ["directory"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write text content to a destination file, creating directories if needed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The path of the destination file to write."
                    },
                    "content": {
                        "type": "string",
                        "description": "The textual content to write into the file."
                    }
                },
                "required": ["filepath", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_in_file",
            "description": "Perform a case-insensitive keyword search in a file and return matching lines with surrounding context.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The file path to search inside."
                    },
                    "keyword": {
                        "type": "string",
                        "description": "The keyword or phrase to search for."
                    }
                },
                "required": ["filepath", "keyword"]
            }
        }
    }
]

print(f"Registered {len(tools)} tool schemas with the LLM:")
for t in tools:
    params = list(t["function"]["parameters"]["properties"].keys())
    print(f"  - {t['function']['name']}({', '.join(params)})")

Registered 4 tool schemas with the LLM:
  - read_file(filepath)
  - list_files(directory, extension)
  - write_file(filepath, content)
  - search_in_file(filepath, keyword)


### Step 6 — Tool Execution Dispatcher

A dispatcher function maps the LLM's requested function name and arguments to the actual Python implementation.

In [14]:
def execute_tool_call(tool_call):
    """
    Parse the LLM's tool call request and dispatch to the correct Python function.

    Args:
        tool_call: An OpenAI tool call object with .function.name and .function.arguments.

    Returns:
        str: JSON-serialized result from the tool function.
    """
    function_name = tool_call.function.name
    try:
        arguments = json.loads(tool_call.function.arguments)
    except Exception as e:
        return json.dumps({"status": "failed", "error": f"Invalid arguments: {e}"})

    print(f"  [Tool Call]  {function_name}({json.dumps(arguments)})")

    try:
        if function_name == "read_file":
            result = read_file(arguments.get("filepath"))
        elif function_name == "list_files":
            result = list_files(arguments.get("directory"), arguments.get("extension"))
        elif function_name == "write_file":
            result = write_file(arguments.get("filepath"), arguments.get("content"))
        elif function_name == "search_in_file":
            result = search_in_file(arguments.get("filepath"), arguments.get("keyword"))
        else:
            result = {"status": "failed", "error": f"Unknown tool: {function_name}"}
    except Exception as e:
        result = {"status": "failed", "error": str(e)}

    # Log a short preview of the result
    preview = str(result)
    if len(preview) > 200:
        preview = preview[:200] + " ...[truncated]"
    print(f"  [Tool Result] {preview}")

    return json.dumps(result)


print("Tool execution dispatcher defined.")

Tool execution dispatcher defined.


### Step 7 — Multi-Step LLM Agentic Loop

The assistant loop:
1. Sends a user query to the LLM with available tool schemas.
2. If the LLM requests tool calls, executes them and appends results to the message history.
3. Repeats until the LLM produces a final natural language response (no more tool calls).

This pattern supports **multi-step** agentic workflows (e.g. `list_files` → `search_in_file` → `write_file`).

In [15]:
def run_assistant(query: str, max_iterations: int = 10) -> str:
    """
    Run an LLM assistant with multi-step tool calling support.

    The LLM iteratively calls tools until it can formulate a final answer.

    Args:
        query (str): Natural language user query.
        max_iterations (int): Max tool-calling rounds before forcing a stop.

    Returns:
        str: Final response from the LLM.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI File System Assistant with tools to read, list, search, and write files. "
                "When asked to perform file operations on resumes or other documents, always use the appropriate tools "
                "to retrieve real data before giving your response. Be thorough, structured, and helpful."
            )
        },
        {"role": "user", "content": query}
    ]

    print("=" * 65)
    print(f"User Query: {query}")
    print("=" * 65)

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if tool_calls:
            # Append LLM tool-call request to message history
            messages.append(response_message)

            # Execute all requested tool calls
            for tool_call in tool_calls:
                tool_result_json = execute_tool_call(tool_call)
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": tool_call.function.name,
                    "content": tool_result_json
                })
        else:
            # LLM produced a final answer — no more tool calls needed
            final_answer = response_message.content
            print(f"\nAssistant Response:\n{'=' * 65}")
            print(final_answer)
            print("=" * 65 + "\n")
            return final_answer

    return "[Max tool iterations reached]"


print("LLM assistant loop defined.")

LLM assistant loop defined.


---
## 🧪 Step 8 — Demo Query 1: Read All Resumes

**Query:** *"Read all resumes in the resumes folder"*

**Expected LLM Tool Call Flow:**
1. `list_files(directory="resumes")` → gets list of all resume files
2. `read_file(filepath=...)` → called for each resume
3. LLM summarizes all resumes in a structured response

In [16]:
response_1 = run_assistant("Read all resumes in the resumes folder")

User Query: Read all resumes in the resumes folder
  [Tool Call]  list_files({"directory": "resumes"})
  [Tool Result] [{'name': 'resume_alice_harper.pdf', 'path': 'resumes\\resume_alice_harper.pdf', 'size_bytes': 2144, 'modified_date': '2026-07-23 06:27:42 UTC', 'type': '.pdf'}, {'name': 'resume_bob_williams.pdf', 'p ...[truncated]
  [Tool Call]  read_file({"filepath": "resumes\\resume_alice_harper.pdf"})
  [Tool Result] {'status': 'success', 'content': 'Alice Harper - Professor of English Literature & Literary\nCritic\nProfessional Summary:\nScholar of 19th-century British literature, Victorian fiction, and narrative ...[truncated]
  [Tool Call]  read_file({"filepath": "resumes\\resume_bob_williams.pdf"})
  [Tool Result] {'status': 'success', 'content': 'Bob Williams - DevOps & Cloud Infrastructure\nEngineer\nProfessional Summary:\nDevOps Engineer with deep expertise in cloud infrastructure automation, Kubernetes orch ...[truncated]
  [Tool Call]  read_file({"filepath": "resumes\\res

---
## 🔍 Step 9 — Demo Query 2: Find Resumes Mentioning Python

**Query:** *"Find resumes mentioning Python experience"*

**Expected LLM Tool Call Flow:**
1. `list_files(directory="resumes")` → discover all resume files
2. `search_in_file(filepath=..., keyword="Python")` → for each resume
3. LLM reports which candidates mention Python and where

In [17]:
response_2 = run_assistant("Find resumes mentioning Python experience")

User Query: Find resumes mentioning Python experience
  [Tool Call]  list_files({"directory": "resumes", "extension": ".pdf"})
  [Tool Result] [{'name': 'resume_alice_harper.pdf', 'path': 'resumes\\resume_alice_harper.pdf', 'size_bytes': 2144, 'modified_date': '2026-07-23 06:27:42 UTC', 'type': '.pdf'}, {'name': 'resume_bob_williams.pdf', 'p ...[truncated]
  [Tool Call]  search_in_file({"filepath": "resumes\\resume_alice_harper.pdf", "keyword": "Python"})
  [Tool Result] {'status': 'success', 'filepath': 'resumes\\resume_alice_harper.pdf', 'keyword': 'Python', 'matches_found': 0, 'matches': []}
  [Tool Call]  search_in_file({"filepath": "resumes\\resume_bob_williams.pdf", "keyword": "Python"})
  [Tool Result] {'status': 'success', 'filepath': 'resumes\\resume_bob_williams.pdf', 'keyword': 'Python', 'matches_found': 3, 'matches': [{'line_number': 5, 'match': 'CI/CD pipelines, and Python scripting.', 'contex ...[truncated]
  [Tool Call]  search_in_file({"filepath": "resumes\\resume_clara

---
## 📝 Step 10 — Demo Query 3: Create a Summary File

**Query:** *"Create a summary file for resumes/resume_john_doe.pdf at summaries/john_doe_summary.txt"*

**Expected LLM Tool Call Flow:**
1. `read_file(filepath="resumes/resume_john_doe.pdf")` → extract content
2. LLM composes a formatted summary
3. `write_file(filepath="summaries/john_doe_summary.txt", content=...)` → save to disk

In [18]:
response_3 = run_assistant(
    "Create a summary file for resumes/resume_john_doe.pdf at summaries/john_doe_summary.txt"
)

User Query: Create a summary file for resumes/resume_john_doe.pdf at summaries/john_doe_summary.txt
  [Tool Call]  read_file({"filepath": "resumes/resume_john_doe.pdf"})
  [Tool Result] {'status': 'success', 'content': 'John Doe - Senior Software Engineer\nProfessional Summary:\nExperienced Software Engineer specializing in backend infrastructure, REST API design, and\ndistributed sy ...[truncated]
  [Tool Call]  write_file({"filepath": "summaries/john_doe_summary.txt", "content": "**John Doe - Senior Software Engineer**\n\n**Professional Summary:**\nExperienced Software Engineer specializing in backend infrastructure, REST API design, and distributed systems. Expert in Python, FastAPI, and Docker.\n\n**Technical Skills:**\nPython, FastAPI, Docker, PostgreSQL, AWS, Redis, Git\n\n**Work Experience:**\n- *Senior Python Developer - TechCorp Innovations (2021 - Present)*  \nArchitected high-throughput microservices using Python and FastAPI. Reduced API latency by 40% using Redis caching an

In [20]:
# Verify the generated summary file was written to disk
summary_path = Path("summaries/john_doe_summary.txt")
if summary_path.exists():
    print(f"Summary file created: {summary_path}")
    print(f"Size: {summary_path.stat().st_size} bytes\n")
    print("Contents:")
    print("-" * 60)
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("Summary file not found — check if write_file was called.")

Summary file created: summaries\john_doe_summary.txt
Size: 766 bytes

Contents:
------------------------------------------------------------
**John Doe - Senior Software Engineer**

**Professional Summary:**
Experienced Software Engineer specializing in backend infrastructure, REST API design, and distributed systems. Expert in Python, FastAPI, and Docker.

**Technical Skills:**
Python, FastAPI, Docker, PostgreSQL, AWS, Redis, Git

**Work Experience:**
- *Senior Python Developer - TechCorp Innovations (2021 - Present)*  
Architected high-throughput microservices using Python and FastAPI. Reduced API latency by 40% using Redis caching and optimized PostgreSQL queries.

- *Software Engineer - CloudScale Systems (2018 - 2021)*  
Built containerized data pipelines with Python and Docker on AWS ECS.

**Education:**
B.S. in Computer Science, University of California, Berkeley (2018)



---
## ✅ Step 11 — Tool Function Validation Tests

Standalone validation tests verifying each tool function works correctly across all supported file formats.

In [21]:
def run_tests():
    """Run a suite of validation checks across all four tool functions."""
    passed = 0
    failed = 0

    def check(name, condition, details=""):
        nonlocal passed, failed
        if condition:
            print(f"  PASS  {name}")
            passed += 1
        else:
            print(f"  FAIL  {name}  {details}")
            failed += 1

    print("=" * 55)
    print("Running Tool Validation Tests")
    print("=" * 55)

    # --- list_files tests ---
    print("\n[list_files]")
    files_all = list_files("resumes")
    check("list_files returns a list", isinstance(files_all, list))
    check("list_files finds 5 resumes", len(files_all) == 5, f"got {len(files_all)}")
    check("each result has name + path", all("name" in f and "path" in f for f in files_all))
    pdf_only = list_files("resumes", ".pdf")
    check("extension filter .pdf works", all(f["name"].endswith(".pdf") for f in pdf_only))

    # --- read_file tests ---
    print("\n[read_file]")
    r_pdf = read_file("resumes/resume_john_doe.pdf")
    check("read PDF: status=success", r_pdf["status"] == "success")
    check("read PDF: content contains 'John Doe'", "john doe" in r_pdf.get("content", "").lower())
    check("read PDF: metadata has modified_date", "modified_date" in r_pdf.get("metadata", {}))

    r_miss = read_file("resumes/nonexistent.pdf")
    check("read missing file: status=failed", r_miss["status"] == "failed")

    r_bad = read_file("resumes/resume_john_doe.pdf".replace(".pdf", ".xyz"))
    check("read unsupported type: status=failed", r_bad["status"] == "failed")

    # --- write_file tests ---
    print("\n[write_file]")
    test_content = "Test write content."
    w = write_file("test_output/nested/test.txt", test_content)
    check("write_file: status=success", w["status"] == "success")
    check("write_file: file exists on disk", Path("test_output/nested/test.txt").exists())
    readback = Path("test_output/nested/test.txt").read_text(encoding="utf-8")
    check("write_file: content matches", readback == test_content)

    # --- search_in_file tests ---
    print("\n[search_in_file]")
    s = search_in_file("resumes/resume_john_doe.pdf", "Python")
    check("search: status=success", s["status"] == "success")
    check("search: found matches", s["matches_found"] > 0, f"got {s['matches_found']}")
    check("search: match has line_number", "line_number" in s["matches"][0])
    check("search: match has context", "context" in s["matches"][0])

    s_none = search_in_file("resumes/resume_john_doe.pdf", "ZZZNotAKeywordZZZ")
    check("search: no match returns 0", s_none["matches_found"] == 0)

    # Cleanup test files
    import shutil
    if Path("test_output").exists():
        shutil.rmtree("test_output")

    print(f"\n{'=' * 55}")
    print(f"Results: {passed} passed, {failed} failed")
    print("=" * 55)


run_tests()

Running Tool Validation Tests

[list_files]
  PASS  list_files returns a list
  FAIL  list_files finds 5 resumes  got 11
  PASS  each result has name + path
  PASS  extension filter .pdf works

[read_file]
  PASS  read PDF: status=success
  PASS  read PDF: content contains 'John Doe'
  PASS  read PDF: metadata has modified_date
  PASS  read missing file: status=failed
  PASS  read unsupported type: status=failed

[write_file]
  PASS  write_file: status=success
  PASS  write_file: file exists on disk
  PASS  write_file: content matches

[search_in_file]
  PASS  search: status=success
  PASS  search: found matches
  PASS  search: match has line_number
  PASS  search: match has context
  PASS  search: no match returns 0

Results: 16 passed, 1 failed


---
## 💬 Step 12 — Interactive Mode 

Run the assistant interactively. Enter any natural language query and watch the LLM invoke tools to answer it. Type `exit` to stop.

In [22]:
print("Interactive LLM File System Assistant")
print("Type 'exit' to stop.\n")

while True:
    try:
        query = input("Your query: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\nSession ended.")
        break

    if not query:
        continue
    if query.lower() in ("exit", "quit"):
        print("Session ended.")
        break

    run_assistant(query)

Interactive LLM File System Assistant
Type 'exit' to stop.

User Query: list all the documents properly
  [Tool Call]  list_files({"directory": "resumes"})
  [Tool Result] [{'name': 'resume_alice_harper.pdf', 'path': 'resumes\\resume_alice_harper.pdf', 'size_bytes': 2144, 'modified_date': '2026-07-23 06:27:42 UTC', 'type': '.pdf'}, {'name': 'resume_bob_williams.pdf', 'p ...[truncated]

Assistant Response:
Here is the list of documents in the "resumes" directory:

### PDF Resumes
1. **[resume_alice_harper.pdf](resumes/resume_alice_harper.pdf)**
   - Size: 2,144 bytes
   - Modified Date: July 23, 2026
2. **[resume_bob_williams.pdf](resumes/resume_bob_williams.pdf)**
   - Size: 2,122 bytes
   - Modified Date: July 23, 2026
3. **[resume_clara_bennett.pdf](resumes/resume_clara_bennett.pdf)**
   - Size: 2,126 bytes
   - Modified Date: July 23, 2026
4. **[resume_david_miller.pdf](resumes/resume_david_miller.pdf)**
   - Size: 2,086 bytes
   - Modified Date: July 23, 2026
5. **[resume_elena_ros